## PgCollection versions (technical documentation)

This page describes PgCollection's different versions, corresponding database structures and version updating (see the section below the table).

The version of PgCollection mainly depends on the structure of the following database tables:

* `'{collection_name}'` -- the collection table. This table stores Text objects of the collection along with their attached layers (layers that are always present when Text objects are retrieved from the database);
* `'{collection_name}__structure'` -- the structure table. This table describes all layers of the collection (contains a row for each layer). Gives information about which of the layers are attached to Text objects (in the collection table), and which of the layers are stored in separate tables (detached and fragmented layers).

<hr>

| version |  `'{collection_name}'` <br> columns | `'{collection_name}__structure'` <br> columns | changes |
|-----------| ----------- | ----------- | ----------- |
| 0.0  | `id BIGSERIAL`, <br> `data jsonb`, <br> `meta columns (optional)` <br>  | `layer_name text`, <br> `detached bool`, <br> `attributes text[]`, <br> `ambiguous bool`, <br> `parent text`, <br> `enveloping text`, <br> `_base text,`, <br> `meta text[]` <br> |--- |
| 1.0 | `id BIGSERIAL`, <br> `data jsonb`, <br> `meta columns (optional)` <br>  | `layer_name text`, <br> `layer_type text`, <br> `attributes text[]`, <br> `ambiguous bool`, <br> `parent text`, <br> `enveloping text`, <br> `_base text,`, <br> `meta text[]` <br> |* changed `detached bool` to `layer_type text` in the structure table to allow storing three types of layers (attached, detached, fragmented); |
| 2.0 | `id BIGSERIAL`, <br> `data jsonb`, <br> `meta columns (optional)` <br>  | `layer_name text`, <br> `attributes text[]`, <br> `ambiguous bool`, <br> `parent text`, <br> `enveloping text`, <br> `meta text[]`, <br> `layer_type text`, <br> `serialisation_module text` | JSON serialization of `Layer` objects was changed, which also required changes in the structure table: <br> * removed `_base` column; <br> * added `serialisation_module` column; <br> * relocated to `layer_type` column; <br> |
| 3.0 |  `id BIGSERIAL`, <br> `data jsonb`, <br> `meta columns (optional)` <br>  | `layer_name text`, <br> `attributes text[]`, <br> `ambiguous bool`, <br> `sparse bool`, <br> `parent text`, <br> `enveloping text`, <br> `meta text[]`, <br> `layer_type text`, <br> `serialisation_module text`, <br> `layer_template jsonb` | Enabled creating sparse layer tables. Sparse layer tables do not store empty layers, which can save up the storage space and allow faster queries over tables and collection: <br> * added `sparse` to structure table (indicates whether the layer is sparse or not?); <br> * added `layer_template` to structure table (template of a default empty layer);  |
| 4.0 |  `id BIGSERIAL`, <br> `data jsonb`, <br> `hidden bool`, <br> `meta columns (optional)` <br> | `layer_name text`, <br> `attributes text[]`, <br>`span_names text[]`, <br> `ambiguous bool`, <br> `sparse bool`, <br> `parent text`, <br> `enveloping text`, <br> `meta text[]`, <br> `layer_type text`, <br> `serialisation_module text`, <br> `layer_template jsonb` | * added column `hidden` to collection table, which can be used to mark some documents being hidden from queries. Note: a row-level security policy must be defined for the hiding to take effect; <br> * added column `span_names` to the structure table; this enabled storing of relation layers in the database; |

<hr>

### PgCollection version updating

EstNLTK contains function `update_structure` for updating PgCollection to a new version. 
Currently, only specific update paths have been implemented. 

#### Updating from {2.0, 3.0} to 4.0

```python 
from estnltk.storage.postgres import PostgresStorage
from estnltk.storage.postgres.structure.update_structure import update_structure

# Connect to the storage
storage = PostgresStorage(pgpass_file='~/.pgpass', 
                          schema=my_schema, 
                          dbname=my_db_name)

# Update collection to the version 4.0
update_structure(storage, collection_name, '4.0')
# (note: if the collection is large, this will take 
#  time because the collection table will be rewritten)

# Restart the connection after the update
storage.close()
# Reconnect
storage = PostgresStorage(pgpass_file='~/.pgpass', 
                          schema=my_schema, 
                          dbname=my_db_name)

# Check version
collection = storage[collection_name]
assert collection.version == '4.0'
```